In [1]:
!pip -q install transformers sentencepiece sacremoses accelerate tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 39.2 MB/s eta 0:00:00


In [2]:
import os
import re
import gc
import math
import shutil
from pathlib import Path

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from google.colab import drive

In [3]:
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
# =========================================
# 2. FIXED PATH TRÊN DRIVE
#    Bạn chỉ cần giữ nguyên folder này là được
# =========================================
BASE_DIR = Path("/content/drive/MyDrive/helsinki_resume_translate")
BASE_DIR.mkdir(parents=True, exist_ok=True)

INPUT_CSV_PATH = BASE_DIR / "processed data.csv"
PROGRESS_CSV_PATH = BASE_DIR / "translation_progress.csv"
OUTPUT_CSV_PATH = BASE_DIR / "processed_data_translated_en.csv"

# Nếu file chưa nằm trong Drive nhưng đang có sẵn trong /content thì tự copy vào Drive
temp_candidates = [
    Path("/content/processed data.csv"),
    Path("/content/processed_data.csv")
]
if not INPUT_CSV_PATH.exists():
    copied = False
    for cand in temp_candidates:
        if cand.exists():
            shutil.copy2(cand, INPUT_CSV_PATH)
            print(f"Đã copy file input vào Drive: {INPUT_CSV_PATH}")
            copied = True
            break
    if not copied:
        raise FileNotFoundError(
            f"Không thấy file input tại: {INPUT_CSV_PATH}\n"
            f"Bạn hãy upload file gốc vào đúng folder này trên Drive 1 lần."
        )

In [5]:
# =========================================
# 3. CẤU HÌNH
# =========================================
ROW_BATCH_SIZE = 24        # số comment xử lý mỗi đợt
CHUNK_BATCH_SIZE = 48      # số chunk nhỏ xử lý mỗi lần generate
MAX_INPUT_LENGTH = 512
MAX_NEW_TOKENS = 256
NUM_BEAMS = 2              # tăng lên 4 nếu muốn chất lượng cao hơn nhưng chậm hơn
SAVE_FULL_EVERY = 20       # mỗi 20 batch sẽ rebuild file output đầy đủ 1 lần

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [6]:
# =========================================
# 4. MAP NGÔN NGỮ -> MODEL HELSINKI-NLP
# =========================================
MODEL_MAP = {
    "vi": "Helsinki-NLP/opus-mt-vi-en",
    "ko": "Helsinki-NLP/opus-mt-ko-en",
    "fr": "Helsinki-NLP/opus-mt-fr-en",
    "de": "Helsinki-NLP/opus-mt-de-en",
    "ru": "Helsinki-NLP/opus-mt-ru-en",
    "es": "Helsinki-NLP/opus-mt-es-en",
    "it": "Helsinki-NLP/opus-mt-it-en",
    "nl": "Helsinki-NLP/opus-mt-nl-en",
    "ja": "Helsinki-NLP/opus-mt-ja-en",
    "zh": "Helsinki-NLP/opus-mt-zh-en",
}


In [7]:
# =========================================
# 5. HÀM TIỆN ÍCH
# =========================================
def find_column(df, candidates):
    lower_map = {c.strip().lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    raise ValueError(f"Không tìm thấy cột nào trong danh sách: {candidates}")

def normalize_lang(x):
    if pd.isna(x):
        return ""
    return str(x).strip().lower()

def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def split_long_text(text, max_chars=450):
    """
    Chia comment dài thành đoạn nhỏ hơn để giảm nguy cơ bị truncate.
    """
    text = clean_text(text)
    if not text:
        return []

    if len(text) <= max_chars:
        return [text]

    parts = re.split(r'(?<=[\.\!\?。！？…])\s+|\n+', text)
    parts = [p.strip() for p in parts if p and p.strip()]

    chunks = []
    current = ""

    for part in parts:
        if len(part) > max_chars:
            # part quá dài thì cắt cứng
            if current:
                chunks.append(current.strip())
                current = ""
            for i in range(0, len(part), max_chars):
                chunks.append(part[i:i+max_chars].strip())
            continue

        if not current:
            current = part
        elif len(current) + 1 + len(part) <= max_chars:
            current = current + " " + part
        else:
            chunks.append(current.strip())
            current = part

    if current:
        chunks.append(current.strip())

    return chunks

def load_model_and_tokenizer(model_name):
    print(f"\nĐang load model: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    model.to(DEVICE)
    model.eval()
    return tokenizer, model

def release_model(model=None, tokenizer=None):
    del model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def translate_chunk_batch(texts, tokenizer, model):
    if not texts:
        return []

    encoded = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_LENGTH
    )
    encoded = {k: v.to(DEVICE) for k, v in encoded.items()}

    with torch.no_grad():
        generated = model.generate(
            **encoded,
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=NUM_BEAMS,
            early_stopping=True
        )

    decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
    return [x.strip() for x in decoded]

def translate_text_list(text_list, tokenizer, model):
    """
    text_list: list comment gốc
    -> split thành chunk
    -> dịch chunk theo batch
    -> ghép lại
    """
    mapping = []
    flat_chunks = []

    for text in text_list:
        chunks = split_long_text(text)
        idxs = []
        for ch in chunks:
            idxs.append(len(flat_chunks))
            flat_chunks.append(ch)
        mapping.append(idxs)

    flat_translations = []
    for start in range(0, len(flat_chunks), CHUNK_BATCH_SIZE):
        sub = flat_chunks[start:start + CHUNK_BATCH_SIZE]
        out = translate_chunk_batch(sub, tokenizer, model)
        flat_translations.extend(out)

    results = []
    for idxs in mapping:
        if not idxs:
            results.append("")
        else:
            merged = " ".join(flat_translations[i] for i in idxs).strip()
            results.append(merged)

    return results

def load_progress_dedup():
    if not PROGRESS_CSV_PATH.exists():
        return pd.DataFrame(columns=["__row_id", "Comment_en", "translation_status", "translation_model"])

    progress = pd.read_csv(PROGRESS_CSV_PATH)
    if progress.empty:
        return pd.DataFrame(columns=["__row_id", "Comment_en", "translation_status", "translation_model"])

    progress["__row_id"] = progress["__row_id"].astype(int)
    progress = progress.drop_duplicates(subset=["__row_id"], keep="last").copy()
    return progress

def append_progress(batch_df):
    header = not PROGRESS_CSV_PATH.exists()
    batch_df.to_csv(PROGRESS_CSV_PATH, mode="a", header=header, index=False, encoding="utf-8-sig")

def rebuild_full_output(df, lang_col, comment_col):
    progress = load_progress_dedup()

    merged = df.merge(progress, on="__row_id", how="left")

    lang_norm = merged[lang_col].map(normalize_lang)
    text_norm = merged[comment_col].map(clean_text)

    # comment rỗng
    mask_empty = text_norm.eq("")
    merged.loc[mask_empty & merged["Comment_en"].isna(), "Comment_en"] = ""
    merged.loc[mask_empty & merged["translation_status"].isna(), "translation_status"] = "empty_comment"
    merged.loc[mask_empty & merged["translation_model"].isna(), "translation_model"] = "none"

    # comment đã là tiếng Anh
    mask_en = lang_norm.eq("en")
    merged.loc[mask_en & merged["Comment_en"].isna(), "Comment_en"] = merged.loc[mask_en, comment_col]
    merged.loc[mask_en & merged["translation_status"].isna(), "translation_status"] = "already_english"
    merged.loc[mask_en & merged["translation_model"].isna(), "translation_model"] = "copy_original"

    # ngôn ngữ không hỗ trợ
    mask_unsupported = (~lang_norm.eq("en")) & (~lang_norm.isin(MODEL_MAP.keys())) & (~mask_empty)
    merged.loc[mask_unsupported & merged["Comment_en"].isna(), "Comment_en"] = merged.loc[mask_unsupported, comment_col]
    merged.loc[mask_unsupported & merged["translation_status"].isna(), "translation_status"] = "unsupported_language"
    merged.loc[mask_unsupported & merged["translation_model"].isna(), "translation_model"] = "unsupported"

    merged = merged.sort_values("__row_id").drop(columns=["__row_id"])
    merged.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8-sig")

def show_progress_summary(df, lang_col, comment_col):
    progress = load_progress_dedup()

    lang_norm = df[lang_col].map(normalize_lang)
    text_norm = df[comment_col].map(clean_text)

    supported_non_en_mask = lang_norm.isin(MODEL_MAP.keys()) & (~lang_norm.eq("en")) & text_norm.ne("")
    total_need_translate = int(supported_non_en_mask.sum())

    done_ids = set(progress["__row_id"].tolist())
    done_count = int(df["__row_id"].isin(done_ids).sum())

    print("\n========== TIẾN ĐỘ ==========")
    print(f"Tổng số dòng cần dịch thật sự (không tính en): {total_need_translate}")
    print(f"Số dòng đã dịch và lưu progress: {done_count}")
    print(f"Còn lại: {max(total_need_translate - done_count, 0)}")
    print(f"File progress: {PROGRESS_CSV_PATH}")
    print(f"File output:   {OUTPUT_CSV_PATH}")
    print("================================\n")

In [8]:
# =========================================
# 6. ĐỌC FILE
# =========================================
df = pd.read_csv(INPUT_CSV_PATH, low_memory=False)
df["__row_id"] = range(len(df))

LANG_COL = find_column(df, ["language", "lang", "Language", "Lang"])
COMMENT_COL = find_column(df, ["comment", "Comment", "text", "review", "content"])

df[LANG_COL] = df[LANG_COL].map(normalize_lang)
df[COMMENT_COL] = df[COMMENT_COL].map(clean_text)

print("Cột language:", LANG_COL)
print("Cột comment :", COMMENT_COL)
print("Shape:", df.shape)

print("\nPhân bố ngôn ngữ:")
print(df[LANG_COL].value_counts(dropna=False).head(20))

Cột language: Language
Cột comment : Comment
Shape: (64407, 7)

Phân bố ngôn ngữ:
Language
en    49219
vi     8001
ko     1746
fr     1353
de     1171
ru      822
es      699
it      506
nl      434
ja      292
zh      164
Name: count, dtype: int64


In [9]:
# =========================================
# 7. RESUME TỰ ĐỘNG
# =========================================
show_progress_summary(df, LANG_COL, COMMENT_COL)

progress = load_progress_dedup()
done_ids = set(progress["__row_id"].tolist())

# chỉ dịch những dòng:
# - comment không rỗng
# - language thuộc MODEL_MAP
# - không phải en
# - chưa có trong progress
translate_mask = (
    df[COMMENT_COL].ne("") &
    df[LANG_COL].isin(MODEL_MAP.keys()) &
    (~df[LANG_COL].eq("en")) &
    (~df["__row_id"].isin(done_ids))
)

langs_to_process = [lang for lang in MODEL_MAP.keys() if (df.loc[translate_mask, LANG_COL] == lang).any()]
print("Ngôn ngữ cần xử lý tiếp:", langs_to_process)


========== TIẾN ĐỘ ==========
Tổng số dòng cần dịch thật sự (không tính en): 15188
Số dòng đã dịch và lưu progress: 72
Còn lại: 15116
File progress: /content/drive/MyDrive/helsinki_resume_translate/translation_progress.csv
File output:   /content/drive/MyDrive/helsinki_resume_translate/processed_data_translated_en.csv

Ngôn ngữ cần xử lý tiếp: ['vi', 'ko', 'fr', 'de', 'ru', 'es', 'it', 'nl', 'ja', 'zh']


In [10]:
# =========================================
# 8. CHẠY DỊCH
# =========================================
batch_counter = 0

for lang in langs_to_process:
    model_name = MODEL_MAP[lang]

    lang_df = df.loc[translate_mask & (df[LANG_COL] == lang), ["__row_id", COMMENT_COL]].copy()
    row_ids = lang_df["__row_id"].tolist()
    texts = lang_df[COMMENT_COL].tolist()

    print(f"\n===== Bắt đầu dịch ngôn ngữ: {lang} | số dòng còn lại: {len(row_ids)} =====")

    tokenizer, model = load_model_and_tokenizer(model_name)

    try:
        for start in range(0, len(row_ids), ROW_BATCH_SIZE):
            batch_row_ids = row_ids[start:start + ROW_BATCH_SIZE]
            batch_texts = texts[start:start + ROW_BATCH_SIZE]

            try:
                batch_translations = translate_text_list(batch_texts, tokenizer, model)
            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    print("CUDA OOM -> dọn bộ nhớ rồi thử lại batch nhỏ hơn")
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
                    gc.collect()

                    # fallback: xử lý từng nửa batch
                    batch_translations = []
                    half = max(1, len(batch_texts) // 2)
                    for s in range(0, len(batch_texts), half):
                        sub_texts = batch_texts[s:s+half]
                        sub_out = translate_text_list(sub_texts, tokenizer, model)
                        batch_translations.extend(sub_out)
                else:
                    raise e

            batch_out = pd.DataFrame({
                "__row_id": batch_row_ids,
                "Comment_en": batch_translations,
                "translation_status": "translated",
                "translation_model": model_name
            })

            append_progress(batch_out)
            batch_counter += 1

            if batch_counter % 5 == 0:
                print(f"Đã lưu progress tới batch {batch_counter} | ngôn ngữ {lang} | dòng {min(start + ROW_BATCH_SIZE, len(row_ids))}/{len(row_ids)}")

            if batch_counter % SAVE_FULL_EVERY == 0:
                rebuild_full_output(df, LANG_COL, COMMENT_COL)
                print("Đã rebuild file output tạm thời")

    finally:
        release_model(model, tokenizer)

    # xong mỗi ngôn ngữ rebuild luôn 1 lần
    rebuild_full_output(df, LANG_COL, COMMENT_COL)
    print(f"Xong ngôn ngữ: {lang}")


===== Bắt đầu dịch ngôn ngữ: vi | số dòng còn lại: 7929 =====

Đang load model: Helsinki-NLP/opus-mt-vi-en


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/756k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Đã lưu progress tới batch 630 | ngôn ngữ zh | dòng 24/164
Đã lưu progress tới batch 635 | ngôn ngữ zh | dòng 144/164
Xong ngôn ngữ: zh


In [11]:
# =========================================
# 9. BUILD FILE CUỐI
# =========================================
rebuild_full_output(df, LANG_COL, COMMENT_COL)
show_progress_summary(df, LANG_COL, COMMENT_COL)

print("Hoàn tất.")
print("Progress CSV:", PROGRESS_CSV_PATH)
print("Final CSV   :", OUTPUT_CSV_PATH)


========== TIẾN ĐỘ ==========
Tổng số dòng cần dịch thật sự (không tính en): 15188
Số dòng đã dịch và lưu progress: 15188
Còn lại: 0
File progress: /content/drive/MyDrive/helsinki_resume_translate/translation_progress.csv
File output:   /content/drive/MyDrive/helsinki_resume_translate/processed_data_translated_en.csv

Hoàn tất.
Progress CSV: /content/drive/MyDrive/helsinki_resume_translate/translation_progress.csv
Final CSV   : /content/drive/MyDrive/helsinki_resume_translate/processed_data_translated_en.csv


In [12]:
out = pd.read_csv("/content/drive/MyDrive/helsinki_resume_translate/processed_data_translated_en.csv", low_memory=False)
print(out.head(10))
print(out.columns.tolist())

   review_id      Username        Address  Rating Language  \
0          1       Mashajo  An Bang Beach       5       ru   
1          2          cj h  An Bang Beach       5       en   
2          3   Hoàng Quỳnh  An Bang Beach       5       vi   
3          4   Snus Hoi An  An Bang Beach       5       en   
4          5     Lavinie A  An Bang Beach       5       en   
5          6  Russ Lacuata  An Bang Beach       5       en   
6          7        Leaver  An Bang Beach       5       en   
7          8  Anabel Kelly  An Bang Beach       5       en   
8          9      Karoly H  An Bang Beach       5       en   
9         10      Fiona MK  An Bang Beach       5       en   

                                             Comment  \
0  Он находится примерно в 10-15 минутах езды от ...   
1              Quiet and quaint. Very laid back vive   
2  Đây là một bãi biển thư giãn tuyệt vời với làn...   
3  While spending time in Hoi An, I was looking f...   
4  Hi everyone—October’s storm caused